ENTREGA DEMANDAS HANDS-OS

GRUPO 10: 
 - Pedro Miranda Rodrigues
 - Gabriel de Castro Lima
 - Ariane Moura Krettli Sant Anna

DADOS OBTIDOS DAS BASES: 
 - Cupons agressivos (>20%) em Instagram Ads e Influencer a partir de Jan/2026
 - Canais como Instagram Ads e Influencer atraem pedidos de item unico focados em Barras e Snacks (40,9%) Bebidas Funcionais (25,6%) e Vitaminas (13,9%), em detrimento dos produtos ancora como Whey (9,9%) e Creatina (9,7%)

HIPOTESE:
 - Canais como Instagram Ads e Influencer canibalizam a receita devido cupons agressivos e vendas de itens unicos (compras pequenas com grandes discontos)

CONSEQUENCIA DA HIPOTESE: 
 - Deixamos de capturar R$ 620,424.56 por ano em receita (considerando 50% de recuperacao via travas de carrinho). A empresa esta subsidiando volume de pedidos com margem.

AÇÕES SUGERIDAS: 
 - Travar cupons com valor minimo (ex: desconto condicional a carrinhos acima de R$ 150).
 - Criar Bundles/Kits exclusivos para trafego pago, diluindo o desconto em um ticket maior.
 - Realinhar KPIs de Marketing: A remuneracao de influenciadores deve ser atrelada a Receita/Margem gerada, e nao a Volume de Pedidos.

<h1 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#3A1156; font-weight:700; font-size:2em; margin-bottom:0.4em;">BootCamp Nova Geração — Análise Exploratória e Diagnóstico com IA</h1>

<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">Case FitNutri: Queda de Ticket Médio</h2>

<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">Contexto de Negócio</h2>

A **FitNutri** é uma rede de 3 lojas de suplementos e alimentação saudável (Centro, Shopping Norte, Barra). Nos últimos 12 meses, o número de pedidos cresceu 22%, mas o **ticket médio caiu 18%** (de R\$ 127 para R\$ 104).

O diretor comercial quer entender:
- **Por que** o ticket médio está caindo se o volume de vendas cresce?
- Essa queda é uniforme ou concentrada em algum corte (loja, categoria, canal de aquisição)?
- Qual a **oportunidade financeira** de reverter o cenário?

<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">Bases de Dados</h2>

| Base | Grão | Período | Descrição |
|------|------|---------|----------|
| `pedidos.csv` | 1 linha por pedido | 12 meses | Receita, itens, loja, cliente, canal |
| `produtos.csv` | 1 linha por produto | — | Categoria, preço de lista, margem |
| `campanhas_marketing.csv` | 1 linha por campanha-mês | 12 meses | Canal, investimento, cupom, desconto médio |

<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">Metodologia</h2>

Siga o ciclo da análise exploratória:
1. **Perfilar** — entender as bases, período, completude
2. **Tratar** — identificar anomalias (pedidos de teste, duplicados)
3. **Explorar** — abrir ticket médio por dimensão (loja, categoria, canal, mês)
4. **Testar hipótese** — verificar se a explicação se sustenta em outros cortes
5. **Sintetizar** — reduzir o achado a uma frase com número

<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">Instruções</h2>

- Use um **agente de IA** (Amazon Q, ChatGPT, Claude, etc.) como analista
- Estruture seus prompts conforme a aula: Papel + Contexto + Dado + Tarefa + Formato + Restrição
- O agente **gera o código**, vocês **validam os números e interpretam**
- Entrega: relatório curto com 2 achados + 1 hipótese testada + dimensionamento da oportunidade


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

df_pedidos = pd.read_csv('pedidos.csv', sep=',')
df_produtos = pd.read_csv('produtos.csv', sep=',')
df_campanhas = pd.read_csv('campanhas_marketing.csv', sep=',')

---
<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">1. Perfilar as Bases</h2>

**Objetivo:** Entender tamanho, período, colunas e qualidade.

Sugestão de prompt para o agente de IA:

> Papel: Analista de dados de varejo.  
> Contexto: Rede de suplementos, 3 lojas, 12 meses de vendas, em reais.  
> Dado: pedidos.csv (1 linha por pedido), produtos.csv, campanhas_marketing.csv.  
> Tarefa: Perfilar cada base — período coberto, nulos, tipos de colunas, distribuição das variáveis numéricas.  
> Formato: Tabela resumo por base + alertas de qualidade.  
> Restrição: Não inferir dados ausentes.


In [ ]:
# Conversão (coerce transforma erros de parse em NaN)
df_pedidos['data_pedido'] = pd.to_datetime(df_pedidos['data_pedido'], errors='coerce')
df_pedidos['n_itens'] = pd.to_numeric(df_pedidos['n_itens'], errors='coerce')
df_pedidos['receita_liquida'] = pd.to_numeric(df_pedidos['receita_liquida'], errors='coerce')
df_pedidos['desconto_aplicado'] = pd.to_numeric(df_pedidos['desconto_aplicado'], errors='coerce')

# Remoção de registros inválidos (ex: linhas de teste ou com erro de checkout)
df_pedidos = df_pedidos.dropna(subset=['data_pedido', 'receita_liquida'])

# Perfilamento Limpo (Relatório Executivo)
print("="*60)
print("RELATORIO DE INTEGRIDADE: BASE DE PEDIDOS")
print("="*60)
print(f"Total de Registros Validos: {len(df_pedidos):,}")
print(f"Periodo Coberto: {df_pedidos['data_pedido'].min().date()} a {df_pedidos['data_pedido'].max().date()}")
print("-"*60)
print(f"{'Coluna':<25} | {'Nulos':<10} | {'Tipo de Dado'}")
print("-"*60)
for col in df_pedidos.columns:
    nulos = df_pedidos[col].isnull().sum()
    tipo = str(df_pedidos[col].dtype)
    print(f"{col:<25} | {nulos:<10} | {tipo}")

print("\n" + "="*60)
print("ESTATISTICAS DESCRITIVAS (Variaveis Criticas)")
print("="*60)
stats = df_pedidos[['n_itens', 'receita_liquida', 'desconto_aplicado']].describe().loc[['mean', 'min', 'max']]
print(f"{'Metrica':<15} | {'Itens (Med)':<12} | {'Receita (R$)':<15} | {'Desconto (%)':<15}")
print("-"*60)
for idx in stats.index:
    i = stats.loc[idx, 'n_itens']
    r = stats.loc[idx, 'receita_liquida']
    d = stats.loc[idx, 'desconto_aplicado'] * 100
    label = {'mean': 'Media', 'min': 'Minimo', 'max': 'Maximo'}[idx]
    print(f"{label:<15} | {i:<12.2f} | {r:<15.2f} | {d:<15.2f}")


RELATORIO DE INTEGRIDADE: BASE DE PEDIDOS
Total de Registros Validos: 13,835
Periodo Coberto: 2025-07-01 a 2026-06-27
------------------------------------------------------------
Coluna                    | Nulos      | Tipo de Dado
------------------------------------------------------------
pedido_id                 | 0          | str
data_pedido               | 0          | datetime64[us]
loja                      | 0          | str
canal_aquisicao           | 0          | str
n_itens                   | 0          | int64
categoria_principal       | 0          | str
receita_liquida           | 0          | float64
desconto_aplicado         | 0          | float64
cliente_id                | 0          | str

ESTATISTICAS DESCRITIVAS (Variaveis Criticas)
Metrica         | Itens (Med)  | Receita (R$)    | Desconto (%)   
------------------------------------------------------------
Media           | 2.12         | 174.83          | 8.65           
Minimo          | 1.00         | 7.21 

---
<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">2. Explorar — Abrir o Ticket Médio por Dimensão</h2>

**Objetivo:** Identificar onde a queda se concentra.

Dimensões sugeridas:
- Ticket médio **por mês** (série temporal)
- Ticket médio **por loja**
- Ticket médio **por canal de aquisição**
- Ticket médio **por categoria principal**
- Distribuição de **desconto aplicado** antes e depois do mês 7


In [ ]:
# Engenharia de Features
df_pedidos['ticket_medio'] = df_pedidos['receita_liquida']
df_pedidos['mes'] = df_pedidos['data_pedido'].dt.to_period('M')

df_pedidos['semestre'] = np.where(df_pedidos['data_pedido'].dt.year == 2025, 'S1 (Jul-Dez 25)', 'S2 (Jan-Jun 26)')

def imprimir_tabela(titulo, serie):
    print(f"\n--- {titulo.upper()} ---")
    print(f"{'Dimensao':<35} | {'Ticket Medio (R$)'}")
    print("-"*45)
    for idx, val in serie.items():
        print(f"{str(idx):<35} | R$ {val:>10.2f}")

# 1. Série Temporal
tm_mes = df_pedidos.groupby('mes')['ticket_medio'].mean().round(2)
imprimir_tabela("Evolucao Mensal", tm_mes)

# 2. Por Loja
tm_loja = df_pedidos.groupby('loja')['ticket_medio'].mean().round(2)
imprimir_tabela("Ticket Medio por Loja", tm_loja)

# 3. Por Canal de Aquisicao
tm_canal = df_pedidos.groupby('canal_aquisicao')['ticket_medio'].mean().round(2)
imprimir_tabela("Ticket Medio por Canal", tm_canal)

# 4. Por Categoria Principal
tm_cat = df_pedidos.groupby('categoria_principal')['ticket_medio'].mean().round(2)
imprimir_tabela("Ticket Medio por Categoria", tm_cat)

# 5. Evolução do Desconto Médio (S1 vs S2)
desc_sem = df_pedidos.groupby('semestre')['desconto_aplicado'].mean() * 100
print("\n--- EVOLUCAO DO DESCONTO MEDIO (S1 vs S2) ---")
for sem, val in desc_sem.items():
    print(f"{sem:<20} : {val:.2f}%")


--- EVOLUCAO MENSAL ---
Dimensao                            | Ticket Medio (R$)
---------------------------------------------
2025-07                             | R$     221.25
2025-08                             | R$     220.36
2025-09                             | R$     213.63
2025-10                             | R$     226.89
2025-11                             | R$     217.71
2025-12                             | R$     217.71
2026-01                             | R$     145.39
2026-02                             | R$     139.07
2026-03                             | R$     139.98
2026-04                             | R$     143.54
2026-05                             | R$     142.85
2026-06                             | R$     143.65

--- TICKET MEDIO POR LOJA ---
Dimensao                            | Ticket Medio (R$)
---------------------------------------------
Barra                               | R$     173.34
Centro                              | R$     174.54
Shopping Nor

---
<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">3. Testar Hipótese</h2>

**Objetivo:** Verificar se a explicação se sustenta em outros cortes.


In [ ]:
# Cruzamento: Semestre vs Canal (Volume, Ticket e Desconto)
analise = df_pedidos.groupby(['semestre', 'canal_aquisicao']).agg(
    volume=('pedido_id', 'count'),
    ticket=('ticket_medio', 'mean'),
    desconto=('desconto_aplicado', 'mean'),
    itens=('n_itens', 'mean')
).round(2)
analise['desconto'] = analise['desconto'] * 100

print("--- CRUZAMENTO: CANAL DE AQUISICAO POR SEMESTRE ---")
print(f"{'Semestre':<18} | {'Canal':<15} | {'Volume':<8} | {'Ticket':<10} | {'Desc(%)':<10} | {'Itens':<8}")
print("-"*75)
for idx, row in analise.iterrows():
    sem, canal = idx
    print(f"{sem:<18} | {canal:<15} | {row['volume']:<8} | R$ {row['ticket']:<8.2f} | {row['desconto']:<8.1f}% | {row['itens']:<8.1f}")

# Foco no ponto de ruptura: Canais de Mídia no S2 (Jan-Jun 2026)
s2_pago = df_pedidos[
    (df_pedidos['semestre'] == 'S2 (Jan-Jun 26)') & 
    (df_pedidos['canal_aquisicao'].isin(['Instagram Ads', 'Influencer']))
]

print("\n--- MIX DE CATEGORIAS (Apenas Instagram/Influencer no S2) ---")
mix = s2_pago['categoria_principal'].value_counts(normalize=True) * 100
print(f"{'Categoria':<25} | {'Share de Pedidos (%)'}")
print("-"*35)
for cat, share in mix.items():
    print(f"{cat:<25} | {share:>10.1f}%")

--- CRUZAMENTO: CANAL DE AQUISICAO POR SEMESTRE ---
Semestre           | Canal           | Volume   | Ticket     | Desc(%)    | Itens   
---------------------------------------------------------------------------
S1 (Jul-Dez 25)    | Busca Paga      | 1482.0   | R$ 220.07   | 4.0     % | 2.3     
S1 (Jul-Dez 25)    | CRM/Email       | 1116.0   | R$ 219.35   | 4.0     % | 2.3     
S1 (Jul-Dez 25)    | Influencer      | 552.0    | R$ 223.99   | 4.0     % | 2.4     
S1 (Jul-Dez 25)    | Instagram Ads   | 901.0    | R$ 222.45   | 4.0     % | 2.3     
S1 (Jul-Dez 25)    | Orgânico        | 1753.0   | R$ 216.76   | 4.0     % | 2.3     
S2 (Jan-Jun 26)    | Busca Paga      | 1204.0   | R$ 216.93   | 4.0     % | 2.3     
S2 (Jan-Jun 26)    | CRM/Email       | 1184.0   | R$ 221.23   | 4.0     % | 2.4     
S2 (Jan-Jun 26)    | Influencer      | 2060.0   | R$ 66.23    | 20.0    % | 1.6     
S2 (Jan-Jun 26)    | Instagram Ads   | 1979.0   | R$ 66.07    | 20.0    % | 1.6     
S2 (Jan-Jun 26)    | O

---
<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">4. Dimensionar a Oportunidade</h2>

**Objetivo:** Estimar o impacto financeiro caso o ticket médio volte ao patamar anterior.

Fórmula: **Base Afetada × Alavanca × Taxa de Captura = Oportunidade**



In [32]:
# 1. Base Afetada: Volume de pedidos ocorridos no S2
base_afetada = len(df_pedidos[df_pedidos['semestre'] == 'S2 (Jan-Jun 26)'])

# 2. Alavanca: A perda de valor por pedido (Ticket S1 - Ticket S2)
tm_s1 = df_pedidos[df_pedidos['semestre'] == 'S1 (Jul-Dez 25)']['ticket_medio'].mean()
tm_s2 = df_pedidos[df_pedidos['semestre'] == 'S2 (Jan-Jun 26)']['ticket_medio'].mean()
alavanca = tm_s1 - tm_s2 

# 3. Taxa de Captura: Premissa conservadora de negócio (50% de recuperação via travas de carrinho/bundles)
taxa_captura = 0.50 

# Cálculo
oportunidade_semestral = base_afetada * alavanca * taxa_captura
oportunidade_anual = oportunidade_semestral * 2

print("="*55)
print("DIMENSIONAMENTO FINANCEIRO DA OPORTUNIDADE")
print("="*55)
print(f"Base Afetada (Pedidos S2) : {base_afetada:>10,} pedidos")
print(f"Ticket Medio S1 (Baseline): R$ {tm_s1:>9.2f}")
print(f"Ticket Medio S2 (Atual)   : R$ {tm_s2:>9.2f}")
print(f"Alavanca (Perda/Pedido)   : R$ {alavanca:>9.2f}")
print(f"Taxa de Captura Estimada  : {taxa_captura:>10.0%}")
print("-"*55)
print(f"Oportunidade Semestral    : R$ {oportunidade_semestral:>12,.2f}")
print(f"Oportunidade Anualizada   : R$ {oportunidade_anual:>12,.2f}")
print("="*55)

DIMENSIONAMENTO FINANCEIRO DA OPORTUNIDADE
Base Afetada (Pedidos S2) :      8,031 pedidos
Ticket Medio S1 (Baseline): R$    219.68
Ticket Medio S2 (Atual)   : R$    142.42
Alavanca (Perda/Pedido)   : R$     77.25
Taxa de Captura Estimada  :        50%
-------------------------------------------------------
Oportunidade Semestral    : R$   310,212.28
Oportunidade Anualizada   : R$   620,424.56


---
<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">5. Sintetizar e Entregar</h2>

Resuma em formato de diagnóstico:

| Item | Descrição |
|------|-----------|
| **Fato** | O ticket médio caiu X% de R\$ Y para R\$ Z |
| **Causa** | [Sua hipótese confirmada com número] |
| **Implicação** | R\$ ??? por ano em receita que deixa de ser capturada |
| **Ação recomendada** | [O que fazer para reverter] |

<h2 style="font-family:'PP Telegraf', 'Helvetica Neue', Arial, sans-serif; color:#12213B; font-weight:700; font-size:1.5em; margin-top:1.2em; margin-bottom:0.4em;">Critérios de aceite</h2>

- [ ] Todo número é rastreável à base
- [ ] Premissas estão listadas
- [ ] O achado muda uma decisão concreta


In [33]:
queda_pct = ((tm_s2 / tm_s1) - 1) * 100

markdown_table = f"""
### Diagnostico FitNutri: Queda de Ticket Medio e Canibalizacao de Receita

| Item | Descricao |
|------|-----------|
| **Fato** | O volume de pedidos cresceu, mas o ticket medio caiu **{abs(queda_pct):.1f}%** (de R$ {tm_s1:.2f} no S1 para R$ {tm_s2:.2f} no S2). |
| **Causa** | A ativacao de **cupons agressivos (>20%)** em *Instagram Ads* e *Influencer* a partir de Jan/2026 canibalizou a receita. Esses canais passaram a atrair pedidos de **item unico** (focados em *Bebidas Funcionais* e *Barras e Snacks*), em detrimento dos produtos ancora (Whey/Creatina). |
| **Implicacao** | Deixamos de capturar **R$ {oportunidade_anual:,.2f}** por ano em receita (considerando 50% de recuperacao via travas de carrinho). A empresa esta subsidiando volume de pedidos com margem. |
| **Acao recomendada** | 1. **Travar cupons com valor minimo** (ex: desconto condicional a carrinhos acima de R$ 150).<br>2. **Criar Bundles/Kits** exclusivos para trafego pago, diluindo o desconto em um ticket maior.<br>3. **Realinhar KPIs de Marketing**: A remuneracao de influenciadores deve ser atrelada a *Receita/Margem* gerada, e nao a *Volume de Pedidos*. |
"""
from IPython.display import display, Markdown
display(Markdown(markdown_table))


### Diagnostico FitNutri: Queda de Ticket Medio e Canibalizacao de Receita

| Item | Descricao |
|------|-----------|
| **Fato** | O volume de pedidos cresceu, mas o ticket medio caiu **35.2%** (de R$ 219.68 no S1 para R$ 142.42 no S2). |
| **Causa** | A ativacao de **cupons agressivos (>20%)** em *Instagram Ads* e *Influencer* a partir de Jan/2026 canibalizou a receita. Esses canais passaram a atrair pedidos de **item unico** (focados em *Bebidas Funcionais* e *Barras e Snacks*), em detrimento dos produtos ancora (Whey/Creatina). |
| **Implicacao** | Deixamos de capturar **R$ 620,424.56** por ano em receita (considerando 50% de recuperacao via travas de carrinho). A empresa esta subsidiando volume de pedidos com margem. |
| **Acao recomendada** | 1. **Travar cupons com valor minimo** (ex: desconto condicional a carrinhos acima de R$ 150).<br>2. **Criar Bundles/Kits** exclusivos para trafego pago, diluindo o desconto em um ticket maior.<br>3. **Realinhar KPIs de Marketing**: A remuneracao de influenciadores deve ser atrelada a *Receita/Margem* gerada, e nao a *Volume de Pedidos*. |
